In [1]:
import pandas as pd
import os
from pathlib import Path

In [21]:
df = pd.read_json("wikipedia_paragraph_paraphrase.jsonl", lines=True)
df.to_csv("wikipedia_paragraph_paraphrase.csv", index=False)

print("Conversion complete.")

Conversion complete.


In [22]:
df_en_wiki = pd.read_csv("wikipedia_paragraph_paraphrase.csv")

df_en_wiki.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21811 entries, 0 to 21810
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   21811 non-null  int64  
 1   input                21811 non-null  object 
 2   output               21811 non-null  object 
 3   lexical_similarity   21811 non-null  float64
 4   semantic_similarity  21811 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 852.1+ KB


In [23]:
df_en_wiki.head()

,id,input,output,lexical_similarity,semantic_similarity
0,1,Perhaps the first public statement on the matt...,Catherine the Great's 1767 Nakaz included a st...,0.600368,0.735477
1,2,"Over the next several decades, the death penal...","Over the coming decades, the death penalty was...",0.823580,0.706949
2,3,"Chester Himes was born in Jefferson City, Miss...","Chester Himes was born on July 29, 1909, in Je...",0.714724,0.718920
3,4,The Euallomyces life cycle is an anisogamous a...,The Euallomyces life cycle alternates between ...,0.831009,0.787660
4,5,"Humenik joined the PGA Tour in 1989, gaining h...",Humenik joined the PGA Tour in 1989 after gain...,0.835732,0.694707


In [24]:
# func to count words
def count_words(text):
    if isinstance(text, str):
        return len(text.split())
    return 0

In [25]:
# Create word count column
df_en_wiki["word_count"] = df_en_wiki["output"].apply(count_words)

# Filter rows with 200 or more words
df_en_wiki_length_filtered = df_en_wiki[df_en_wiki["word_count"] >= 200].copy()

In [26]:
df_en_wiki_length_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4262 entries, 3 to 21807
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   4262 non-null   int64  
 1   input                4262 non-null   object 
 2   output               4262 non-null   object 
 3   lexical_similarity   4262 non-null   float64
 4   semantic_similarity  4262 non-null   float64
 5   word_count           4262 non-null   int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 233.1+ KB


In [27]:
# Keep only 'output' column and rename it to 'text'
df_en_wiki_length_filtered = (
    df_en_wiki_length_filtered[["output"]]
    .rename(columns={"output": "text"})
    .copy()
)

# Add new column with value 1
df_en_wiki_length_filtered["generated"] = 1

In [28]:
df_en_wiki_length_filtered.head()

,text,generated
3,The Euallomyces life cycle alternates between ...,1
8,"From 2000 to 2015, Glion Institute of Higher E...",1
12,Since the late 1980s and continuing into the p...,1
18,"A turn in the game is called an 'inning', wher...",1
20,Yule's research into the prehistory of Oman st...,1


In [31]:
df_en_wiki_length_filtered.to_csv("filtered_separate/wikipedia_paragraph_paraphrase.csv", index=False)

## PART 2

In [32]:
input_file = "en_wiki_paraphrased.jsonl"
output_file = "temp/en_wiki_paraphrased.csv"

chunk_size = 10000  # reduce to 5000 if RAM becomes tight

with pd.read_json(input_file, lines=True, chunksize=chunk_size) as reader:
    for i, chunk in enumerate(reader):
        if i == 0:
            chunk.to_csv(output_file, index=False, mode="w")
        else:
            chunk.to_csv(output_file, index=False, header=False, mode="a")

print("Conversion complete.")

Conversion complete.


In [36]:
input_file = "temp/en_wiki_paraphrased.csv"
output_file = "filtered_separate/en_wiki_paraphrased.csv"

chunk_size = 10000  # reduce to 5000 if RAM is tight

# def count_words(text):
#     if isinstance(text, str):
#         return len(text.split())
#     return 0

first_chunk = True

for chunk in pd.read_csv(input_file, chunksize=chunk_size):
    
    # Create word count column
    chunk["word_count"] = chunk["paraphrase"].apply(count_words)
    
    # Filter rows
    filtered = chunk[chunk["word_count"] >= 100]
    
    if not filtered.empty:
        # Keep only required column and rename
        filtered = (
            filtered[["paraphrase"]]
            .rename(columns={"paraphrase": "text"})
        )
        
        # Add new column
        filtered["generated"] = 1
        
        # Write or append
        if first_chunk:
            filtered.to_csv(output_file, index=False, mode="w")
            first_chunk = False
        else:
            filtered.to_csv(output_file, index=False, header=False, mode="a")

print("Filtering complete.")

Filtering complete.


In [37]:
input_file = "filtered_separate/en_wiki_paraphrased.csv"
output_file = "filtered_separate/en_wiki_paraphrased_combined.csv"

chunk_size = 10000
buffer_row = None
first_write = True

def word_count(text):
    return len(text.split()) if isinstance(text, str) else 0

for chunk in pd.read_csv(input_file, chunksize=chunk_size):

    texts = chunk["text"].tolist()
    
    if buffer_row is not None:
        texts.insert(0, buffer_row)
        buffer_row = None

    combined_rows = []

    for i in range(0, len(texts), 2):
        if i + 1 < len(texts):
            combined_text = texts[i] + " " + texts[i + 1]

            if word_count(combined_text) >= 200:
                combined_rows.append({
                    "text": combined_text,
                    "generated": 1
                })
        else:
            buffer_row = texts[i]

    if combined_rows:
        combined_df = pd.DataFrame(combined_rows)

        if first_write:
            combined_df.to_csv(output_file, index=False, mode="w")
            first_write = False
        else:
            combined_df.to_csv(output_file, index=False, header=False, mode="a")

print("Combining complete.")

Combining complete.


Final 10k paraphrased AI

In [38]:
# 1. Load the first CSV and get 5738 rows
file1_path = 'filtered_separate/en_wiki_paraphrased_combined.csv'
# nrows loads only the necessary rows, saving memory
df1 = pd.read_csv(file1_path, nrows=5738)

# 2. Load the second CSV
file2_path = 'filtered_separate/wikipedia_paragraph_paraphrase.csv'
df2 = pd.read_csv(file2_path)

# 3. Combine the files (stacking them vertically)
# ignore_index=True reindexes the final dataframe from 0 to N
combined_df = pd.concat([df1, df2], ignore_index=True)

# 4. Save the result to a new CSV file
combined_df.to_csv('paraphrased_wiki_10k.csv', index=False)

print(f"Successfully combined {len(df1)} rows from file 1 and {len(df2)} rows from file 2.")
print(f"Total rows in combined file: {len(combined_df)}")


Successfully combined 5738 rows from file 1 and 4262 rows from file 2.
Total rows in combined file: 10000
